In [7]:
import os

In [8]:
import pandas as pd

In [9]:
import duckdb

In [10]:
df = pd.read_csv("data/raw/contracts.csv", encoding="latin1", low_memory=False)

In [11]:
df['contract_date'] = pd.to_datetime(df['contract_date'], errors='coerce')

In [12]:
df['fiscal_year'] = df['contract_date'].apply(
    lambda d: f"{d.year}-{d.year+1}" if pd.notna(d) and d.month >= 4 else (f"{d.year-1}-{d.year}" if pd.notna(d) else "Unknown")
)

In [13]:
df['contract_value'] = pd.to_numeric(df['contract_value'], errors='coerce').fillna(0)

In [14]:
def clean_department_string(text):
    if not isinstance(text, str) or pd.isna(text):
        return "Unknown Department"
    
    # Split the string by the pipe symbol to get just the English side
    english_part = text.split('|')[0].strip()
    
    # Use standard English fragments to map to perfect, clean bilingual titles
    if "Northern Ontario" in english_part:
        return "Federal Economic Development Agency for Northern Ontario | Agence fédérale de développement économique pour le Nord de l'Ontario"
    elif "Health Canada" in english_part or "Sant" in english_part:
        return "Health Canada | Santé Canada"
    elif "Global Affairs" in english_part:
        return "Global Affairs Canada | Affaires mondiales Canada"
    elif "Immigration, Refugees" in english_part:
        return "Immigration, Refugees and Citizenship Canada | Immigration, Réfugiés et Citoyenneté Canada"
    elif "Innovation, Science" in english_part:
        return "Innovation, Science and Economic Development Canada | Innovation, Sciences et Développement économique Canada"
    elif "Auditor General" in english_part:
        return "Office of the Auditor General of Canada | Bureau du vérificateur général du Canada"
    elif "Privacy Commissioner" in english_part:
        return "Office of the Privacy Commissioner of Canada | Commissariat à la protection de la vie privée du Canada"
    elif "Superintendent of Financial" in english_part:
        return "Office of the Superintendent of Financial Institutions | Bureau du surintendant des institutions financières"
    elif "Pacific Economic" in english_part:
        return "Pacific Economic Development Canada | Développement économique Canada pour le Pacifique"
    elif "Parks Canada" in english_part:
        return "Parks Canada | Parcs Canada"
    elif "Border Services" in english_part:
        return "Canada Border Services Agency | Agence des services frontaliers du Canada"
    
    # If any other department slips through, just use the clean English name side
    return english_part

In [9]:
df['owner_org_title'] = df['owner_org_title'].apply(clean_department_string)

In [30]:
df['vendor_clean'] = (
    df['vendor_name']
    .str.strip()
    .str.rstrip('.')
    .str.upper()
    .str.replace(r'\s+', ' ', regex=True)
)

In [16]:
con = duckdb.connect()

In [17]:
con.execute("DROP TABLE IF EXISTS contracts")

In [18]:
con.execute("CREATE TABLE contracts AS SELECT * FROM df")

In [19]:
df.to_csv("data/clean/contracts_clean.csv", index=False, encoding="utf-8")

In [20]:
print("Data processing complete! All departments standardized.")

Data processing complete! All departments standardized.


In [21]:
df_chart1 = con.execute("""
    SELECT 
        owner_org_title AS department,
        CASE 
            WHEN commodity_type = 'C' THEN 'Competitive'
            WHEN commodity_type = 'S' THEN 'Sole-Source / Non-Competitive'
            WHEN commodity_type = 'G' THEN 'Standing Offer / Supply Arrangement'
            ELSE 'Other / Unspecified'
        END AS procurement_strategy, 
        SUM(contract_value) AS total_value
    FROM contracts
    WHERE department != 'Unknown Department'
    GROUP BY department, procurement_strategy
    ORDER BY department ASC, total_value DESC
""").df()

In [22]:
df_chart1.to_csv("data/clean/chart1_competitive_vs_sole_source.csv", index=False)

In [23]:
df_chart1.head(10)

,department,procurement_strategy,total_value
0,Accessibility Standards Canada | Normes dâac...,Sole-Source / Non-Competitive,2.429754e+07
1,Accessibility Standards Canada | Normes dâac...,Standing Offer / Supply Arrangement,1.113598e+05
2,Administrative Tribunals Support Service of Ca...,Sole-Source / Non-Competitive,2.092487e+08
3,Administrative Tribunals Support Service of Ca...,Standing Offer / Supply Arrangement,3.889337e+07
4,Administrative Tribunals Support Service of Ca...,Competitive,4.004814e+04
5,Agriculture and Agri-Food Canada | Agriculture...,Sole-Source / Non-Competitive,2.051862e+09
6,Agriculture and Agri-Food Canada | Agriculture...,Standing Offer / Supply Arrangement,3.394894e+08
7,Agriculture and Agri-Food Canada | Agriculture...,Competitive,9.617784e+07
8,Agriculture and Agri-Food Canada | Agriculture...,Other / Unspecified,2.696199e+07
9,Atlantic Canada Opportunities Agency | Agence ...,Other / Unspecified,6.860855e+07


In [31]:
df_chart2 = con.execute("""
    SELECT vendor_name, SUM(contract_value) AS total_value, COUNT(*) AS n_contracts
    FROM contracts
    GROUP BY vendor_name
    ORDER BY total_value DESC
    LIMIT 10
    """
                       ).df()

In [25]:
con.execute("PRAGMA table_info('contracts')").df()[['name', 'type']]

,name,type
0,ï»¿reference_number,VARCHAR
1,procurement_id,VARCHAR
2,vendor_name,VARCHAR
3,vendor_postal_code,VARCHAR
4,buyer_name,VARCHAR
5,contract_date,TIMESTAMP_NS
6,economic_object_code,VARCHAR
7,description_en,VARCHAR
8,description_fr,VARCHAR
9,contract_period_start,VARCHAR


In [32]:
df_chart2.to_csv("data/clean/chart2_Top_10_vendors_by_total_value.csv", index=False)

In [33]:
df_chart2.head()

,vendor_name,total_value,n_contracts
0,VANCOUVER SHIPYARDS CO LTD,2.586129e+10,150
1,Irving Shipbuilding Inc,2.563714e+10,6
2,SkyAlyne Canada Limited Partnership,2.243874e+10,2
3,Irving Shipbuilding Inc.,1.876377e+10,2
4,I.M.P Group Limited,1.689638e+10,9


In [29]:
df_chart3 = con.execute("""
    SELECT fiscal_year, MONTH(contract_date) AS month, SUM(contract_value) AS total_value, COUNT(*) AS n_contracts
    FROM contracts
    GROUP BY fiscal_year, month
"""
).df()

In [33]:
df_chart3.to_csv("data/clean/chart3_Monthly_award_volume.csv", index=False)

In [32]:
df_chart3.head()

,fiscal_year,month,total_value,n_contracts
0,2019-2020,2,1.098216e+09,6514
1,2013-2014,11,1.012536e+09,3283
2,2018-2019,11,1.424387e+09,5575
3,2019-2020,10,1.801463e+09,5455
4,2017-2018,4,1.289941e+09,5696


In [35]:
df_chart4 = con.execute("""
WITH shares AS (
  SELECT owner_org_title AS department, vendor_name,
         SUM(contract_value) / SUM(SUM(contract_value)) OVER (PARTITION BY department) AS share
  FROM contracts
  GROUP BY department, vendor_name
)
SELECT department, SUM(POWER(share*100, 2)) AS hhi
FROM shares
GROUP BY department
"""
                       ).df()

In [36]:
df_chart4.to_csv("data/clean/chart4_Vendor_Concentration(HHI)_by_department.csv", index = False)

In [37]:
df_chart4.head()

,department,hhi
0,Agriculture and Agri-Food Canada,152.249026
1,Canadian Northern Economic Development Agency,298.304290
2,Canadian Radio-television and Telecommunicatio...,301.734282
3,Correctional Service of Canada,56.702592
4,Crown-Indigenous Relations and Northern Affair...,89.687772


In [50]:
df_chart5 = con.execute("""
SELECT owner_org_title AS department,
       SUM(contract_value) AS total_value,
       SUM(CASE WHEN LOWER(indigenous_business) IN ('m', 'v', 't', 'true') THEN contract_value ELSE 0 END) AS indigenous_value,
       100.0 * SUM(CASE WHEN LOWER(indigenous_business) IN ('m', 'v', 't', 'true') THEN contract_value ELSE 0 END)
             / NULLIF(SUM(contract_value), 0) AS pct_indigenous
FROM contracts
GROUP BY owner_org_title
ORDER BY pct_indigenous DESC;
"""
                       ).df()

In [51]:
df_chart5.to_csv("data/clean/chart5_Set-aside_performance.csv", index = False)

In [52]:
df_chart5.head()

,department,total_value,indigenous_value,pct_indigenous
0,Atlantic Canada Opportunities Agency,1.056797e+08,0.0,0.0
1,Canada Energy Regulator,3.371634e+08,0.0,0.0
2,Canadian Space Agency,4.112948e+10,0.0,0.0
3,National Gallery of Canada,2.513633e+07,0.0,0.0
4,Farm Products Council of Canada,1.566193e+06,0.0,0.0


In [53]:
con.execute("SELECT indigenous_business, COUNT(*) FROM contracts GROUP BY indigenous_business").df()

,indigenous_business,count_star()
0,MS,1398
1,VS,3873
2,None,1043304


In [54]:
df_chart5 = con.execute("""
SELECT owner_org_title AS department,
       SUM(contract_value) AS total_value,
       SUM(CASE WHEN indigenous_business IN ('MS', 'VS') THEN contract_value ELSE 0 END) AS indigenous_value,
       100.0 * SUM(CASE WHEN indigenous_business IN ('MS', 'VS') THEN contract_value ELSE 0 END)
             / NULLIF(SUM(contract_value), 0) AS pct_indigenous
FROM contracts
GROUP BY owner_org_title
ORDER BY pct_indigenous DESC;
"""
                       ).df()

In [55]:
df_chart5.to_csv("data/clean/chart5_Set-aside_performance.csv", index = False)

In [56]:
df_chart5.head()

,department,total_value,indigenous_value,pct_indigenous
0,Elections Canada,2.716158e+09,1.963548e+08,7.229138
1,Financial Consumer Agency of Canada,2.720568e+08,1.894229e+07,6.962621
2,Canadian Northern Economic Development Agency,4.712697e+06,2.840834e+05,6.028044
3,Canadian Human Rights Commission,4.217748e+07,2.288140e+06,5.425027
4,Administrative Tribunals Support Service of Ca...,2.481821e+08,1.302118e+07,5.246622


In [61]:
df_chart6 = con.execute("""
SELECT 
    CASE LEFT(vendor_postal_code, 1)
        WHEN 'K' THEN 'Ontario (ON)'
        WHEN 'L' THEN 'Ontario (ON)'
        WHEN 'M' THEN 'Ontario (ON)'
        WHEN 'N' THEN 'Ontario (ON)'
        WHEN 'P' THEN 'Ontario (ON)'
        WHEN 'V' THEN 'British Columbia (BC)'
        WHEN 'A' THEN 'Newfoundland and Labrador (NL)'
        WHEN 'B' THEN 'Nova Scotia (NS)'
        WHEN 'C' THEN 'Prince Edward Island (PE)'
        WHEN 'E' THEN 'New Brunswick (NB)'
        WHEN 'G' THEN 'Quebec (QC)'
        WHEN 'H' THEN 'Quebec (QC)'
        WHEN 'J' THEN 'Quebec (QC)'
        WHEN 'R' THEN 'Manitoba (MB)'
        WHEN 'S' THEN 'Saskatchewan (SK)'
        WHEN 'T' THEN 'Alberta (AB)'
        WHEN 'X' THEN 'Northwest Territories / Nunavut (NT/NU)'
        WHEN 'Y' THEN 'Yukon (YT)'
        ELSE 'Other / Unknown Location'
    END AS province,
    ROUND(SUM(contract_value), 2) AS total_value,
    ROUND(100.0 * SUM(contract_value) / SUM(SUM(contract_value)) OVER (), 2) AS pct_of_total
FROM contracts
WHERE vendor_postal_code IS NOT NULL AND vendor_postal_code != ''
GROUP BY province
ORDER BY total_value DESC;
""").df()

In [62]:
df_chart6.to_csv("data/clean/chart6_Regional(provincial)_vendor_distribution.csv", index = False)

In [63]:
df_chart6.head()

,province,total_value,pct_of_total
0,Ontario (ON),1.194918e+11,45.15
1,British Columbia (BC),5.708649e+10,21.57
2,Quebec (QC),4.220205e+10,15.94
3,Nova Scotia (NS),2.603026e+10,9.83
4,Saskatchewan (SK),5.278050e+09,1.99


In [66]:
df_chart7 = con.execute("""
SELECT owner_org_title AS department,
       ROUND(AVG(DATE_DIFF('day', CAST(contract_date AS TIMESTAMP), TRY_CAST(delivery_date AS TIMESTAMP))), 1) AS avg_days,
       MEDIAN(DATE_DIFF('day', CAST(contract_date AS TIMESTAMP), TRY_CAST(delivery_date AS TIMESTAMP))) AS median_days,
       COUNT(*) AS n_contracts
FROM contracts
WHERE TRY_CAST(delivery_date AS TIMESTAMP) > CAST(contract_date AS TIMESTAMP)
GROUP BY owner_org_title
ORDER BY avg_days DESC
""").df()

In [67]:
df_chart7.to_csv("data/clean/chart7_Average_days_from_award_to_delivery_by_department.csv", index = False)

In [68]:
df_chart7.head()

,department,avg_days,median_days,n_contracts
0,Canadian Space Agency,872.9,476.0,4237
1,Canadian Transportation Agency,727.2,426.0,1241
2,Immigration and Refugee Board of Canada,623.9,729.0,5415
3,Employment and Social Development Canada,599.6,236.0,38024
4,"Department of Housing, Infrastructure and Comm...",562.3,363.5,1282


In [70]:
df_chart8 = con.execute("""
SELECT owner_org_title AS department,
       ROUND(SUM(contract_value), 2) AS total_value,
       ROUND(SUM(CASE WHEN trade_agreement IS NULL OR trade_agreement IN ('NA', 'N/A', 'None') THEN contract_value ELSE 0 END), 2) AS exempt_value,
       ROUND(100.0 * SUM(CASE WHEN trade_agreement IS NULL OR trade_agreement IN ('NA', 'N/A', 'None') THEN contract_value ELSE 0 END)
             / NULLIF(SUM(contract_value), 0), 2) AS pct_exempt
FROM contracts
GROUP BY owner_org_title
ORDER BY pct_exempt DESC
""").df()

In [71]:
df_chart8.to_csv("data/clean/chart8_Trade_agreement_coverage.csv", index = False)

In [72]:
df_chart8.head()

,department,total_value,exempt_value,pct_exempt
0,Office of the Taxpayers' Ombudsperson,5.288452e+05,5.288452e+05,100.00
1,Parks Canada | Parcs Canada,8.218670e+09,7.760697e+09,94.43
2,Canada Economic Development for Quebec Regions,9.568773e+07,8.199400e+07,85.69
3,Department of Finance Canada,7.011093e+08,5.891828e+08,84.04
4,Atlantic Canada Opportunities Agency,1.056797e+08,8.661805e+07,81.96


In [73]:
df_chart9 = con.execute("""
WITH pairs AS (
  SELECT a.owner_org_title AS department, 
         a.vendor_name,
         a."ï»¿reference_number" AS ref_a, 
         b."ï»¿reference_number" AS ref_b,
         a.contract_value AS val_a, 
         b.contract_value AS val_b,
         ABS(DATE_DIFF('day', CAST(a.contract_date AS TIMESTAMP), CAST(b.contract_date AS TIMESTAMP))) AS day_gap
  FROM contracts a
  JOIN contracts b
    ON a.owner_org_title = b.owner_org_title
   AND a.vendor_name = b.vendor_name
   AND a."ï»¿reference_number" < b."ï»¿reference_number"
  WHERE ABS(DATE_DIFF('day', CAST(a.contract_date AS TIMESTAMP), CAST(b.contract_date AS TIMESTAMP))) <= 30
    AND a.contract_value < 25000
    AND b.contract_value < 25000
)
SELECT department, 
       vendor_name,
       COUNT(*) AS suspicious_pairs,
       ROUND(SUM(val_a + val_b), 2) AS combined_value
FROM pairs
GROUP BY department, vendor_name
ORDER BY combined_value DESC;
""").df()

In [76]:
df_chart9.to_csv("data/clean/Possible_Contract_Splitting_Patterns.csv", index = False)

In [77]:
df_chart9.head()

,department,vendor_name,suspicious_pairs,combined_value
0,Correctional Service of Canada,MCKESSON CANADA CORPORATION,102174,3.169641e+09
1,Correctional Service of Canada,MCKESSON CANADA,24064,7.348201e+08
2,National Defence,SHELL,10423,3.384992e+08
3,National Defence,AEG FUELS,9642,3.329119e+08
4,Correctional Service of Canada,ARI FINANCIAL SERVICES,8120,2.433419e+08


In [34]:
import os

In [35]:
import pandas as pd

In [36]:
import duckdb

In [37]:
df = pd.read_csv("data/raw/contracts.csv", encoding="latin1", low_memory=False)

In [38]:
df['contract_date'] = pd.to_datetime(df['contract_date'], errors='coerce')

In [39]:
df['fiscal_year'] = df['contract_date'].apply(
    lambda d: f"{d.year}-{d.year+1}" if pd.notna(d) and d.month >= 4 else (f"{d.year-1}-{d.year}" if pd.notna(d) else "Unknown")
)

In [40]:
df['contract_value'] = pd.to_numeric(df['contract_value'], errors='coerce').fillna(0)

In [41]:
df_chart2_vendors = con.execute("""
    SELECT 
        CASE 
            WHEN vendor_name LIKE 'Irving Shipbuilding%' THEN 'Irving Shipbuilding Inc.'
            WHEN vendor_name LIKE 'Vancouver Shipyards%' THEN 'Vancouver Shipyards Co.'
            ELSE vendor_name
        END AS vendor_name, 
        ROUND(SUM(contract_value), 2) AS total_value, 
        COUNT(*) AS n_contracts
    FROM contracts
    WHERE vendor_name IS NOT NULL AND vendor_name != ''
    GROUP BY vendor_name
    ORDER BY total_value DESC
    LIMIT 10
""").df()

In [43]:
df_chart2_vendors.head(10)

,vendor_name,total_value,n_contracts
0,VANCOUVER SHIPYARDS CO LTD,2.586129e+10,150
1,Irving Shipbuilding Inc.,2.563714e+10,6
2,SkyAlyne Canada Limited Partnership,2.243874e+10,2
3,Irving Shipbuilding Inc.,1.876377e+10,2
4,I.M.P Group Limited,1.689638e+10,9
5,MDA SYSTEMS LTD.,1.585963e+10,166
6,General Dynamics Land Systems - Canada Corpora...,1.458308e+10,68
7,L-3 Communications MAS (Canada) Inc.,1.330826e+10,104
8,"MACDONALD, DETTWILER & ASSOCIATES LTD.",1.255882e+10,27
9,Lockheed Martin Canada Inc.,1.202905e+10,106


In [44]:
df_chart2_vendors = con.execute("""
    WITH cleaned_vendors AS (
        SELECT 
            CASE 
                WHEN vendor_name LIKE 'Irving Shipbuilding%' THEN 'Irving Shipbuilding Inc.'
                WHEN vendor_name LIKE 'Vancouver Shipyards%' THEN 'Vancouver Shipyards Co.'
                ELSE vendor_name
            END AS vendor_name,
            contract_value
        FROM contracts
        WHERE vendor_name IS NOT NULL AND vendor_name != ''
    )
    SELECT 
        vendor_name, 
        ROUND(SUM(contract_value), 2) AS total_value, 
        COUNT(*) AS n_contracts
    FROM cleaned_vendors
    GROUP BY vendor_name
    ORDER BY total_value DESC
    LIMIT 10
""").df()

In [45]:
df_chart2_vendors.head(10)

,vendor_name,total_value,n_contracts
0,Irving Shipbuilding Inc.,5.192878e+10,26
1,VANCOUVER SHIPYARDS CO LTD,2.586129e+10,150
2,SkyAlyne Canada Limited Partnership,2.243874e+10,2
3,I.M.P Group Limited,1.689638e+10,9
4,MDA SYSTEMS LTD.,1.585963e+10,166
5,General Dynamics Land Systems - Canada Corpora...,1.458308e+10,68
6,L-3 Communications MAS (Canada) Inc.,1.330826e+10,104
7,"MACDONALD, DETTWILER & ASSOCIATES LTD.",1.255882e+10,27
8,Lockheed Martin Canada Inc.,1.202905e+10,106
9,Vancouver Shipyards Co.,1.174632e+10,24


In [46]:
df_chart2_vendors.to_csv("data/clean/Chart2_TopVendors.csv", index=False, encoding="utf-8")